# 🛠️ 06 — The hands, and the invariance bet

*How "ticket #7 says 50 Mbps" becomes actual router state — and what happens to all the
machinery you've built when Bell starts selling something completely different.*

Chapter 05's bouncer said **yes**. Somebody now has to act on that yes: walk over to a
real router and change its settings so Ada's traffic actually flows at 50 Mbps. This
chapter builds those hands — and then springs the question the whole paper turns on:
Bell wants to sell a *second* product, from a different plane of the network entirely.
**How much of chapters 03–05 has to change?** You'll place your bet before we find out.

**You need:** nothing. This chapter runs entirely against the *mock* provisioner (the
recording double the repo's own tests use). Readers with the containerlab lab running
can afterwards replay everything against the real router in
[`provisioner.py`](../../../netctl/src/netctl/provisioner.py) — the same calls, in the shipped provisioner.

**How to work through it:** run every cell in order; on **✏️ Your turn**, try before
opening the solution. **🧭 Decision** boxes and the closing **📝 For the paper** section
work as in chapter 03 — principled decisions argued, pragmatic ones honestly labeled.

## 0 · Where we are, and three words about routers

The story so far: the offer was signed (04), the swap was atomic (03), the bouncer
checked six facts against the chain and said yes (05). What's left is *enforcement* —
the physical world doing what the ticket says.

Three glosses, and you know everything this chapter assumes about networking:

- **A router's configuration is a settings tree.** Not code — *data*. A big nested
  structure of sections and values: interfaces, QoS rules, export targets. Configuring
  a router = writing values into that tree; unconfiguring = deleting them. Think of a
  giant `dict`.
- **YANG** is the schema of that tree — the document that says which branches exist and
  what type each leaf has. You won't read any YANG today; you just need to know the
  tree has a schema, so paths into it are *checkable*.
- **gNMI** is a standard network API for reading and writing that tree from another
  machine: `get(path)`, `set(path, value)`, `subscribe(path)`. A **path** is an address
  into the tree, written like a filesystem path with selectors:
  `/qos/policer-templates/policer-template[name=a2a-7]`.

That's the whole vocabulary. One more piece of naming from the story: the thing Bell
sells for ticket #7 is enforced by a **policer** — a rule in the QoS section of the
tree that says "traffic on this interface may not exceed N bits per second". Commit a
policer, and the router's dataplane throttles; delete it, and the pipe opens back up.

So the naive job description is: *given an entitlement, write a policer into the right
router's tree; at teardown, delete it.* Let's do the naive version and rob it.

## 1 · The naive hands — and the topology leak

First, a router to practice on. A real SR Linux router speaks gNMI; for building the
*reasoning* we only need something that behaves like one — a settings tree with
`set` and `delete`:

In [ ]:
class ToyRouter:
    """A router reduced to what this chapter needs: a config tree you can edit."""

    def __init__(self, name):
        self.name = name
        self.config = {}                      # path -> value: the settings tree, flattened

    def set(self, path, value):
        self.config[path] = value

    def delete(self, path):
        del self.config[path]                 # (note this KeyErrors on a missing path — remember that)

srl1 = ToyRouter("srl1")
print(srl1.name, "booted with an empty config tree:", srl1.config)

And the naive hands. Ticket #7 (chapter 03's finished toy left it in `machine.tickets`;
here's the same dict) says 50 Mbps — so write a policer. A sensible first version
hardcodes what it knows about Bell's network:

In [ ]:
ticket7 = {
    "id": 7, "owner": "Ada", "issuer": "Bell",
    "service_type": 0,                                # 0 = bandwidth (03 §3.6)
    "resource_id": "0x" + "07".rjust(64, "0"),        # the opaque 32-byte handle from the chain
    "capacity_mbps": 50,
    "start": 1_757_944_800, "end": 1_757_952_000,
}

def configure_bandwidth_v1(router, ticket):
    # Bell knows his own network: ticket 7 is the hostA->hostB path through srl1,
    # entering at port ethernet-1/1. Hardcode it.
    name = f"a2a-{ticket['id']}"
    router.set(f"/qos/policer-templates/policer-template[name={name}]",
               {"peak-rate-kbps": ticket["capacity_mbps"] * 1000})
    router.set("/qos/interfaces/interface[interface-id=ethernet-1/1.0]",
               {"policer-template": name})

configure_bandwidth_v1(srl1, ticket7)
for path, value in srl1.config.items():
    print(path, "=", value)

Two writes: the rule itself (a *policer template* named after the session) and its
*attachment* to the ingress interface — the rule does nothing until some port is told
to use it. This is genuinely the shape of the real thing. And yet this function has two
problems worth a robbery each.

**Robbery #1 — the ticket knows too much (or the function does).** Look at where the
knowledge lives. The hardcoded `srl1` / `ethernet-1/1` means this function only works
for ticket #7 — so the *obvious* fix is to put the device and port **into the ticket**:
let `resource_id` be `"srl1:ethernet-1/1"` and the function generic. Tempting. Now
remember chapter 03: the ticket lives **on a public chain**. Everyone — every customer,
every competitor, every attacker — can read it:

In [ ]:
# The "obvious fix": resource_id names the real hardware. On a PUBLIC chain.
leaky_ticket = dict(ticket7, resource_id="srl1:ethernet-1/1")

print("What Mallory learns by reading the public chain:")
print("  - Bell has a device called 'srl1'")
print("  - its customer-facing port is 'ethernet-1/1'")
print("  - every other ticket maps Bell's network further, one sale at a time")
print()
print("And when Bell re-cables srl1 so Ada's path enters at ethernet-1/3 instead:")
print("  - the terms frozen into every sold ticket are now WRONG,")
print("    and 03 §3.6 made them immutable on purpose.")

Both failures come from the same mistake: **a public, immutable ledger is the wrong
home for private, changeable facts.** Bell's topology is both — private (it's his
network's floor plan) and changeable (he re-cables, upgrades, renames).

The fix is a level of indirection. The chain keeps an **opaque handle** — 32 bytes that
mean nothing to anyone but Bell (`0x…07`, and you met it in chapter 03: the contract
stored it without interpreting it). Inside Bell's controller — private, editable — one
small table resolves handles to hardware. The repo calls it the **resource map**:

In [ ]:
# Bell's private notebook: opaque public handle -> concrete private topology.
# Two SHAPES of resolution, because products differ in what they need:
#   a bandwidth ticket needs a PATH  (device + which port in, which port out),
#   (soon) a telemetry ticket needs a NODE (just: which device).

RESOURCE_MAP = {
    "0x" + "07".rjust(64, "0"): {"kind": "path", "device": "srl1",
                                 "ingress_if": "ethernet-1/1", "egress_if": "ethernet-1/2"},
}

def configure_bandwidth_v2(routers, ticket, resource_map):
    resolved = resource_map.get(ticket["resource_id"])
    if resolved is None:
        raise Exception("UnmappedResource: valid ticket, wrong venue")   # more on this below
    router = routers[resolved["device"]]
    name = f"a2a-{ticket['id']}"
    router.set(f"/qos/policer-templates/policer-template[name={name}]",
               {"peak-rate-kbps": ticket["capacity_mbps"] * 1000})
    router.set(f"/qos/interfaces/interface[interface-id={resolved['ingress_if']}.0]",
               {"policer-template": name})

srl1 = ToyRouter("srl1")
configure_bandwidth_v2({"srl1": srl1}, ticket7, RESOURCE_MAP)
for path, value in srl1.config.items():
    print(path, "=", value)

Note the new refusal: a syntactically valid ticket whose handle isn't in *this*
provider's map gets `UnmappedResource` — "valid ticket, wrong venue", like presenting a
genuine Madrid concert ticket at a Barcelona door. (The repo's controller surfaces this
as the deny-code `E_SCOPE`; chapter 05's predicate had a cousin of it.)

> **🧭 Decision (principled) — the resource map lives in the controller, and nowhere else**
>
> **Chosen:** `resourceId` is opaque on-chain; exactly one private file inside the
> provider's controller (`controller/src/controller/resource_map.yaml`) resolves it to
> device + interfaces; the gNMI layer below receives only *concrete names*.
> **Alternatives:** (a) topology in the ticket, on-chain; (b) the resolution table down
> in `netctl`, the gNMI layer itself.
> **Why:** (a) is the robbery you just ran — a public immutable ledger holding private
> changeable facts: topology leaks to every reader, and every re-cable falsifies sold
> tickets. (b) quietly couples the reusable "hands" to one particular lab — netctl
> could no longer be pointed at any topology, and every provider deployment would need
> its own fork. With the map in the controller, topology changes touch exactly one
> file, and the settlement vocabulary (`resourceId`) never leaks downward — nor YANG
> paths upward. The repo enforces this as a hard rule: *netctl is topology-agnostic*.
> **Cost:** one more indirection to explain, and the map is config the provider must
> keep correct — a wrong entry means honoring a ticket on the wrong port.
> **In the paper:** §4.3 ("the resource id is opaque on-chain; a private map inside the
> provider is the only place chain names meet device names") — and it's ADR-005 in the
> repo, alternatives and all.

**✏️ Your turn 1 — Bell re-cables**

Maintenance weekend: Ada's path now enters `srl1` at `ethernet-1/3` (egress unchanged).
Make the *one* edit that reality now requires, re-run the configuration on a fresh
`ToyRouter`, and confirm the attachment moved. Then answer in a comment: what changed
on the chain?

In [ ]:
# your edit + re-run here...

<details><summary>✅ Solution 1 — peek only after trying</summary>

```python
RESOURCE_MAP["0x" + "07".rjust(64, "0")]["ingress_if"] = "ethernet-1/3"
srl1 = ToyRouter("srl1")
configure_bandwidth_v2({"srl1": srl1}, ticket7, RESOURCE_MAP)
print(list(srl1.config))
RESOURCE_MAP["0x" + "07".rjust(64, "0")]["ingress_if"] = "ethernet-1/1"   # put it back
```

One line in one private file. On the chain: **nothing** — ticket #7's bytes are
untouched, which is exactly the point. The immutable half (the promise) and the mutable
half (the floor plan) live on opposite sides of the indirection.

</details>

## 2 · Teardown — and the crash that runs it twice

Enforcement has a mirror image: at the window's end (or on revocation), remove exactly
what was installed. Naively:

In [ ]:
def teardown_v1(routers, ticket, resource_map):
    resolved = resource_map[ticket["resource_id"]]
    router = routers[resolved["device"]]
    name = f"a2a-{ticket['id']}"
    router.delete(f"/qos/policer-templates/policer-template[name={name}]")
    router.delete(f"/qos/interfaces/interface[interface-id={resolved['ingress_if']}.0]")

teardown_v1({"srl1": srl1}, ticket7, RESOURCE_MAP)
print("config after teardown:", srl1.config)

Clean. Now the robbery — except this time the attacker is *reality*, not Mallory.
Chapter 05 left you with two independent triggers that both end a session: the expiry
timer ticking past `end`, and the revocation watcher seeing Bell flip the flag. Suppose
Bell revokes thirty seconds before expiry. Two processes, each doing its duty, each
calling teardown:

In [ ]:
srl1 = ToyRouter("srl1")
configure_bandwidth_v2({"srl1": srl1}, ticket7, RESOURCE_MAP)

teardown_v1({"srl1": srl1}, ticket7, RESOURCE_MAP)        # the watcher got there first
try:
    teardown_v1({"srl1": srl1}, ticket7, RESOURCE_MAP)    # ...and now the expiry timer fires
except KeyError as e:
    print("💥 second teardown crashed on:", e)
    print("The session is actually GONE — the crash is pure noise. But a crash in the")
    print("expiry path looks like a failed teardown, pages a human, poisons the logs...")

The second call found nothing to delete and blew up — yet *the world is exactly as it
should be*. "Remove X" succeeded in every sense that matters; only the naive code
insists on treating "X was already gone" as failure.

The fix is a property with a name you met in chapter 03, on `revoke`: **idempotence** —
doing it twice equals doing it once, and both count as success. Emergency paths
especially must have it, because emergencies are precisely when two panicking processes
press the same button:

In [ ]:
def teardown_v2(routers, ticket, resource_map):
    """Idempotent: 'make it gone' — already-gone is success, not error."""
    resolved = resource_map[ticket["resource_id"]]
    router = routers[resolved["device"]]
    name = f"a2a-{ticket['id']}"
    for path in [f"/qos/policer-templates/policer-template[name={name}]",
                 f"/qos/interfaces/interface[interface-id={resolved['ingress_if']}.0]"]:
        router.config.pop(path, None)        # pop-with-default: delete if present, shrug if not

srl1 = ToyRouter("srl1")
configure_bandwidth_v2({"srl1": srl1}, ticket7, RESOURCE_MAP)
teardown_v2({"srl1": srl1}, ticket7, RESOURCE_MAP)
teardown_v2({"srl1": srl1}, ticket7, RESOURCE_MAP)        # twice. on purpose. calmly.
print("config:", srl1.config, "— torn down twice, zero drama")

Notice the philosophy repeating across layers: `revoke` on the contract re-flips a flag
and succeeds (03 §3.6); teardown at the router re-deletes and succeeds. The repo makes
it a blanket rule — *teardown is idempotent everywhere; calling it twice is a success,
not an error* — because the alternative is every layer defensively wondering whether
some other layer got there first.

One more subtlety hiding in the session name: everything installed for this session is
named `a2a-7`. That's not decoration — it means teardown can *find its own work on the
router itself* instead of trusting some database of "what I think I installed". The
router's config tree is the ground truth about the router. (The real teardown, you'll
see below, literally searches the tree for its name.)

**✏️ Your turn 2 — the half-crash**

Harder failure: `teardown_v1` crashed *between* its two deletes (say, the connection
dropped). The attachment is gone, the template still there. Show that running
`teardown_v2` afterwards heals the half-state — set the scene by hand: configure a
fresh router, delete only the attachment path, then run `teardown_v2` and print the tree.

In [ ]:
# set up the half-torn state, then heal it...

<details><summary>✅ Solution 2 — peek only after trying</summary>

```python
srl1 = ToyRouter("srl1")
configure_bandwidth_v2({"srl1": srl1}, ticket7, RESOURCE_MAP)
del srl1.config["/qos/interfaces/interface[interface-id=ethernet-1/1.0]"]  # the half-crash
teardown_v2({"srl1": srl1}, ticket7, RESOURCE_MAP)
print(srl1.config)   # {} — healed
```

Idempotence buys you more than double-call safety: it makes teardown a **retry-safe
cleanup** — after *any* partial failure, the fix is always simply "run teardown again",
never a bespoke repair per crash site.

</details>

## 3 · The invariance bet

Business is good. Bell decides to sell a second product: **telemetry export**. Ada (or
anyone) pays for the right to have `srl1` *stream its own interface counters* — packets
per second, errors, throughput — to a collector endpoint of the buyer's choosing, every
10 seconds, for a window.

Stop and notice how *alien* this product is compared to bandwidth:

| | bandwidth (ticket #7) | telemetry (ticket #8) |
|---|---|---|
| what's sold | dataplane capacity — how fast *your* traffic moves | a management-plane action — the router *reporting on itself* |
| shaped like | a limit (police to 50 Mbps) | a feed (export counters to 10.0.0.50:57000) |
| terms | capacity, QoS class | sensor paths, collector endpoint, sample interval |
| resolution | a *path* (device + in-port + out-port) | a *node* (just: which device) |

Different plane, different verb, different terms, different shape of resolution.

**✏️ Your turn 3 — place your bet (this one is the chapter)**

Before reading ANY further: go back through chapters 03–05 in your head and write down,
in the scaffold, which of the six lifecycle stages must change to sell this new product,
and roughly how much:

1. discovery & quote (the A2A exchange, 07b's territory)
2. the two LLM judgment slots (accept/reject, quote/decline)
3. settlement (`fulfill` — chapter 03)
4. ownership proof (challenge–response — chapter 05)
5. authorization (the six-check predicate — chapter 05)
6. translate-to-config + teardown (this chapter)

Commit to the bet. The whole paper hangs on this answer.

In [ ]:
# My bet — for each stage, write UNCHANGED or CHANGED (+ what changes):
# 1. discovery/quote:
# 2. LLM slots:
# 3. settlement:
# 4. ownership proof:
# 5. predicate:
# 6. translate + teardown:

<details><summary>✅ Solution 3 — peek only after betting</summary>

Stages 1–5: **unchanged machinery.** New *parameter values* flow through them (a
telemetry need instead of a bandwidth need, different terms in the offer), but not one
mechanism is edited: the same offer struct with the same twelve fields, the same
`fulfill` (chapter 03's contract stored `params` as an opaque blob precisely so it
would never need to understand what it stores — that decision pays off *now*), the same
challenge–response, the same six checks (the predicate reads window/owner/revoked/type
— none of which care what the product does).

Stage 6: **changed** — and *only* here. A new translator (terms → export-destination
write instead of policer write) and its mirror-image teardown.

If your bet said "surely the contract needs a telemetry version" or "the predicate must
check the collector endpoint" — that intuition is exactly what the architecture was
built to defeat, and the next cells make the defeat concrete.

</details>

Now let's *earn* that answer instead of asserting it. Here is the telemetry translator,
built next to the bandwidth one. To make the comparison honest, first restate the
bandwidth hands in the same shape — a pure function from ticket + resolution to a list
of write operations (separating *deciding what to write* from *writing it* — that
separation becomes the real architecture in a moment):

In [ ]:
def translate_bandwidth(ticket, resolved):
    """Terms -> the writes that enforce them. Pure: no router touched."""
    name = f"a2a-{ticket['id']}"
    return [
        ("set", f"/qos/policer-templates/policer-template[name={name}]",
                {"peak-rate-kbps": ticket["capacity_mbps"] * 1000}),
        ("set", f"/qos/interfaces/interface[interface-id={resolved['ingress_if']}.0]",
                {"policer-template": name}),
    ]

def translate_telemetry(ticket, resolved):
    """Same job, different product: terms -> the writes that start the feed."""
    name = f"a2a-{ticket['id']}"
    host, port = ticket["collector_endpoint"].split(":")
    return [
        # The address book: where the buyer's collector lives.
        ("set", f"/system/grpc-tunnel/destination[name={name}]",
                {"address": host, "port": int(port), "network-instance": "mgmt"}),
        # The active half: dials that destination and hands over a gNMI session.
        ("set", f"/system/grpc-tunnel/tunnel[name={name}]",
                {"admin-state": "enable",
                 "destination": [{"name": name}],
                 "target": [{"type": {"gnmi-gnoi-server": "telemetry"}}]}),
    ]

ticket8 = {
    "id": 8, "owner": "Ada", "issuer": "Bell",
    "service_type": 1,                                # 1 = telemetry
    "resource_id": "0x" + "08".rjust(64, "0"),
    "sensor_paths": ["/interface[name=ethernet-1/1]/statistics"],
    "collector_endpoint": "10.0.0.50:57000",
    "sample_interval_s": 10,
    "start": 1_757_944_800, "end": 1_757_952_000,
}
RESOURCE_MAP["0x" + "08".rjust(64, "0")] = {"kind": "node", "device": "srl1"}

for op in translate_bandwidth(ticket7, RESOURCE_MAP[ticket7["resource_id"]]):
    print("bandwidth :", op[0], op[1])
for op in translate_telemetry(ticket8, RESOURCE_MAP[ticket8["resource_id"]]):
    print("telemetry :", op[0], op[1])

Same skeleton — ticket in, named writes out, `a2a-<id>` naming so teardown can find its
work — different branch of the settings tree. The telemetry writes land under
`/system/grpc-tunnel/…` (management plane: the router's own reporting machinery), the
bandwidth writes under `/qos/…` (dataplane: how traffic is treated). Two products from
different planes, one pattern.

And notice the shape repeating: **two writes each, and in both cases only the second one
acts.** A policer template describes a limit; the *attachment* is what makes a port obey
it. A tunnel destination records an address; the *tunnel* is what dials it. Write only
the describing half and you get config that reads back perfectly and does nothing.

That is not a hypothetical. This repo shipped `translate_telemetry` with only the
destination write for three milestones, and every test passed — because every test asked
"is the destination there?", which is a question the inert half can always answer yes to.
It was caught by pointing a real collector at it and waiting: **0 samples in 20 seconds**,
with the tunnel added, counters flowing within one sample interval. The lesson generalizes
past this router: *assert the node that enforces, not the node that describes.*

And now the ledger of what selling ticket #8 actually required. This table is the
paper's Table 2, and you have personally built or run every row of it:

In [ ]:
STAGES = [
    ("discovery, quote",     "same machinery, different params (a telemetry need, not a bandwidth need)"),
    ("LLM accept/decline",   "same two judgment slots, unedited"),
    ("settlement",           "same contract, same fulfill — params stored as an opaque blob (03), never interpreted"),
    ("ownership proof",      "same challenge-response (05)"),
    ("authorization",        "same six-check predicate (05) — owner/window/revoked/type, product-blind"),
    ("translate to config",  "NEW: policer + attachment (qos)  ->  destination + tunnel (grpc-tunnel)"),
    ("teardown",             "NEW: remove what THIS translator installed — mirror image, still idempotent"),
]
width = max(len(s) for s, _ in STAGES)
for stage, verdict in STAGES:
    marker = "CHANGED  " if verdict.startswith("NEW") else "unchanged"
    print(f"{stage:<{width}}  {marker}  {verdict}")

Five stages of shared machinery, two rows of product-specific hands. That ratio is the
**invariance claim** — the paper's RQ2, and its headline result: *the product appears in
exactly one place*. Everything above the translator is a fixed cost, already paid; a
provider adding a product writes one translator and one teardown, nothing else.

Why does this matter enough to headline a paper? Because it's the difference between
"we demoed a bandwidth-vending machine" (an artifact) and "we found the layer boundary
where commerce stops caring what is sold" (an architecture). The evidence is not either
service working — it's their **difference**: change the product across the widest gap
the lab offers (dataplane limit → management-plane feed) and measure what else moved.

**✏️ Your turn 4 — the third product**

Prove you could be Bell's engineer. A customer wants to buy a **firewall pinhole**:
"open TCP port 8443 toward my server 10.0.0.99, for two hours" (assume writes under
`/acl/…`). Sketch `translate_pinhole(ticket, resolved)` in the same shape as the two
above, and answer in comments: which of the seven table rows change? What resolution
kind does it need — path or node?

In [ ]:
# def translate_pinhole(ticket, resolved):
#     ...
# rows that change:
# resolution kind:

<details><summary>✅ Solution 4 — peek only after trying</summary>

```python
def translate_pinhole(ticket, resolved):
    name = f"a2a-{ticket['id']}"
    return [("set",
             f"/acl/acl-filter[name={name}]",
             {"action": "accept", "protocol": "tcp",
              "destination": ticket["server"], "port": ticket["port"]})]
```

Rows that change: the same last two — translate and teardown. Stages 1–5 carry the new
terms without edits (the contract stores `{"server": …, "port": …}` as bytes it never
reads; the predicate still checks owner/window/revoked/type). Resolution kind: a
*path*-flavored answer is defensible (which interface the pinhole opens on), a *node*
one too for a device-wide rule — the point is the map entry decides, privately, and
nothing upstream knows. One translator ≈ a dozen lines: that is the marginal cost of a
new product on this architecture.

</details>

## 4 · Meet the real hands

Everything you built has a production twin, split across two packages along a line you
now understand:

| your toy | the real thing | why the line is there |
|---|---|---|
| `RESOURCE_MAP` dict | `controller/src/controller/resource_map.yaml` + a strict loader | ADR-005: one private file where handles meet hardware |
| `{"kind": "path"/"node"}` | `ResolvedPath` / `ResolvedNode` (typed, frozen shapes from `a2a_interfaces`) | the two shapes of "where" |
| `translate_bandwidth` / `translate_telemetry` | `controller.translators.translate()` — dispatch on `service_type`, one pure function per product | the ONLY product-specific code above the wire |
| `("set", path, value)` tuples | `ProvisionerCall(method=…, kwargs=…)` — *intended* calls, as data | pure translation, applied later at the edge |
| `ToyRouter` | `netctl` — the gNMI provisioner (and `MockProvisioner`, its recording double) | topology-agnostic hands: receive concrete names, speak gNMI |
| `teardown_v2` pop-with-default | `MockProvisioner.teardown` pops with default; the real one *searches the router* for `a2a-<id>` names and deletes what it finds | idempotence, rule 8; the router is the ground truth |
| `Exception("UnmappedResource")` | `translators.UnmappedResource` → surfaced as deny-code `E_SCOPE` | valid ticket, wrong venue |

One design note before running it: your toys returned *operations* and something else
executed them — the real translator does the same, returning `ProvisionerCall` records
that the controller's wiring applies to whatever provisioner it was handed. That's the
chapter-05 trick again (ports and protocols): `translate()` is a pure function
(`controller/domain` imports no I/O — a hard rule), and *which* hands — mock or real
gNMI — is decided at composition time. Today: the mock, the exact double the repo's
contract tests run.

In [ ]:
from a2a_interfaces import fixtures as fx
from controller.resource_map import load_resource_map
from controller.translators import translate, UnmappedResource
from netctl.mock import MockProvisioner

resource_map = load_resource_map()               # the real YAML you saw quoted above
for rid, resolved in resource_map.items():
    print("0x" + rid.hex()[-4:], "→", resolved)

Two entries — tickets #7 and #8, the canonical pair. Now the real translators on the
real canonical entitlements (`fx.CANONICAL_ENTITLEMENT_VIEW` is ticket #7 exactly as
the controller reads it off the chain; `fx.TELEMETRY_ENTITLEMENT_VIEW` is #8):

In [ ]:
calls_bw = translate("sess-7", fx.CANONICAL_ENTITLEMENT_VIEW, resource_map)
calls_tm = translate("sess-8", fx.TELEMETRY_ENTITLEMENT_VIEW, resource_map)

for label, calls in [("bandwidth #7", calls_bw), ("telemetry #8", calls_tm)]:
    for call in calls:
        print(f"{label}:  {call.method}(")
        for k, v in call.kwargs.items():
            print(f"    {k} = {v!r}")
        print(")")

Read the two calls against your toys: the bandwidth one carries a `ResolvedPath` plus
`capacity_bps=50_000_000` (the ticket's stored terms, decoded from the params blob),
the telemetry one a `ResolvedNode` plus sensor paths, collector, interval. Product
knowledge begins and ends here.

Apply them — the wiring's job is one line of `getattr`, so we do exactly what it does —
then tear both down, twice, because we've earned the right to expect calm:

In [ ]:
net = MockProvisioner()      # same Protocol as the real gNMI hands; records instead of doing

for call in calls_bw + calls_tm:
    result = getattr(net, call.method)(**call.kwargs)
    print(call.method, "→", result)

print("\nrouter-side state the mock recorded:")
for session, config in net.applied.items():
    print(" ", session, "→", config)

In [ ]:
for _ in range(2):                       # expiry timer AND revocation watcher, racing
    print("teardown sess-7 →", net.teardown("sess-7"))
print("teardown sess-8 →", net.teardown("sess-8"))

print("\napplied after teardowns:", net.applied)
print("teardown log        :", net.torn_down, " ← the double call, recorded, harmless")

`ok=True` on the second `sess-7` teardown — your `teardown_v2`, verbatim policy. And on
the real router the same promise is kept a stronger way: the real `teardown(session_id)`
doesn't consult memory at all — it asks the router for every template and export
destination named `a2a-<session_id>` and deletes what it finds. Found nothing = nothing
to do = success. The config tree, not the process, is the source of truth.

Why should you believe the mock stands in for the real thing? Because the repo holds
them to it: **mock and real provisioner pass the same contract-test suite** — one set of
tests, parameterized over both implementations, pinning identical behavior at the
Protocol (a repo hard rule: *a mock with different behavior at the port is a bug*). The
notebook you'd use to watch the real one push real gNMI at a live SR Linux is
[`provisioner.py`](../../../netctl/src/netctl/provisioner.py) — the same calls you just made, shipped.

**✏️ Your turn 5 — wrong venue**

Forge an entitlement for a resource Bell never mapped: take
`fx.CANONICAL_ENTITLEMENT_VIEW.model_copy(update={"resource_id": bytes(32)})` (the
all-zero handle) and translate it. Predict the outcome first — is this a crash, or a
*refusal*? Which chapter-03 error is it the cousin of?

In [ ]:
# ghost = fx.CANONICAL_ENTITLEMENT_VIEW.model_copy(update={...})
# ...translate it...

<details><summary>✅ Solution 5 — peek only after trying</summary>

```python
ghost = fx.CANONICAL_ENTITLEMENT_VIEW.model_copy(update={"resource_id": bytes(32)})
try:
    translate("sess-ghost", ghost, resource_map)
except UnmappedResource as e:
    print("refused:", e)
```

A refusal — `UnmappedResource`, surfaced to callers as deny-code `E_SCOPE`: the ticket
may be perfectly valid *somewhere*, but this controller manages no such resource. It's
the venue-side cousin of chapter 03's `WrongConsumer`: both are "genuine credential,
wrong door", and both are deliberate deny-paths, not crashes. (It also quietly closes a
hole: a *different* provider's tickets can't make Bell's controller touch Bell's
routers.)

</details>

## 5 · The honesty section — what "the router enforced it" really means in this lab

This chapter's claims run on a *containerized* SR Linux — Nokia's router OS in a Docker
container, control plane complete, but with a software datapath standing in for the
forwarding ASIC (the specialized chip that, in the hardware product, actually moves and
polices packets at line rate). Measured reality, documented in the repo: the container
**accepts and commits** the policer configuration — and its software datapath **does not
enforce it**. A 100 Mbit/s stream sails through a committed 50 Mbit/s policer.

The repo neither hides this nor quietly patches around it; it documents a **shim**
(ADR-006), and the shim's design is worth reading as an exercise in honest simulation:

- The gNMI-committed policer on the router stays the **single source of truth**.
- A lab script *mirrors* that committed config into a Linux kernel traffic shaper
  (`tc tbf`) **inside the router's own network namespace** — standing exactly where the
  missing ASIC would stand, at the router.
- The controller and netctl **never learn the shim exists**: they speak gNMI to the
  router and nothing else. On hardware, the shim is simply absent and nothing changes.
- Rejected alternative worth naming: having the controller drive `tc` directly would
  have *bypassed the router entirely* — the demo would work and the story
  ("an on-chain ticket configures a real router") would be false.

> **🧭 Decision (pragmatic) — the lab shim stands in for the missing ASIC**
>
> The full-fidelity alternatives — hardware routers with real ASICs, or a router OS
> whose container datapath enforces QoS — were priced and rejected on lab reality
> (licenses, RAM, none viable on the machine; the ADR lists what was tried). The shim
> was chosen because it is *enough to demonstrate the mechanism*: config committed via
> gNMI is what gets enforced, at the router, and the iperf plateau you'll meet in
> chapter 09 (~48 Mbit/s under a 50 Mbit/s ticket) is real physics on that shaper. It
> is **not** a claim that containerized SR Linux polices traffic. The paper states this
> in §5.4 *before* any reviewer can discover it, and repeats it in §8.4: "dataplane
> enforcement uses the documented shim."

For telemetry the lab goes one step further, and the *direction* is the whole argument.

> **🧭 Decision (principled) — the router dials the buyer, and a dial-in collector would prove nothing**
>
> **Chosen:** the lab runs a **collector node** on the management bridge with a
> gRPC-tunnel server; the entitlement's config makes `srl1` dial **out** to it, and
> samples arrive because that config exists. Delete it and the feed stops.
> **Alternatives:** (a) a dial-in collector that subscribes to `srl1` from a lab node;
> (b) a provider-side forwarder process relaying samples.
> **Why:** (a) is evidence-shaped decoration — samples would flow because the collector
> *asked*, which is true whether or not any ticket was ever honored; it tests gNMI, not
> the entitlement. (b) makes delivery depend on a provider process rather than durable
> on-device config, which is the opposite of the claim. Only dial-out makes the
> committed configuration the *cause* of delivery. Measured: destination alone,
> 0 samples in 20 s; destination + tunnel, counters flowing on a 5 s beat; after
> teardown, 0 samples in 20 s.
> **Cost:** the lab needs the collector node, and a CPM firewall entry for the dial-out's
> return path (the router's control-plane filter is a stateless port allow-list applied
> in *both* directions — a lab fixture, like the shim; on hardware the CPM policy is the
> operator's). See ADR-008.

One thing this does **not** buy, and the paper says so: the tunnel hands the collector a
gNMI session, and the entitlement's `sensor_paths` are **not enforced** on it. Scoping a
tunneled principal to specific paths is gNSI *pathz*, which the prototype does not
implement — so a collector could subscribe outside what it bought. Measured and named as
a gap, not quietly left out: the cheaper route (a read-only role) was tried on this image
and denies *every* read, so it is not usable. Chapter 09 returns to what the trust model
assumes versus enforces.

## 6 · What you can now say

- **What the hands are:** a translator (pure: terms → named write-operations) and a
  provisioner (gNMI: apply concrete writes to concrete devices) — with product
  knowledge confined to the translator, topology knowledge confined to one private map,
  and neither leaking into the other.
- **Why the map exists:** a public immutable ledger is the wrong home for private
  changeable facts — you ran both failure modes (topology leak, re-cable rot).
- **Why teardown is idempotent:** two triggers race to end every session; "already
  gone" is success. Same philosophy as `revoke` — at every layer.
- **The invariance result:** selling a second product from a different plane changed
  two rows out of seven — translate and teardown — and you bet on it before you saw it.
- **Describe vs enforce:** both products write a describing node and an acting node, and
  only the second one does anything. A readback that asks after the first can go green
  against configuration that enforces nothing — which is how a real bug hid here for
  three milestones.
- **What the lab honestly shows:** committed gNMI config as the source of truth,
  dataplane enforcement via a documented ASIC-stand-in at the router, and telemetry
  delivered by dial-out so the entitlement's config is the *cause* of the feed —
  with the sensor-path scoping gap named rather than papered over.

**Loose threads, on purpose:** who *calls* translate-then-apply, and where the LLM
does and doesn't sit in that loop — chapters 07a/07b; the measured numbers behind
"the plateau is real" and "gas differs by terms-size only" — chapter 09.

## 7 · 📝 For the paper

Where this chapter lands (`main.tex` numbering): **§IV-C Service Adapters**, **§IV-A**'s
testbed caveats (both honesty notes), **§V-B RQ2** (the result), **§VI-B
Generalizability** (the marginal-cost argument), plus §III-C's one sentence on the
opaque resource id. This is **RQ2's chapter** — the paper's distinguishing claim.

| you can write… | because you ran… |
|---|---|
| *The resource id is opaque on-chain; a private map inside the provider is the only place chain names meet device names.* | §1's topology-leak robbery, the one-line re-cable fix, and the real `resource_map.yaml` loaded in §4 |
| *Swapping a dataplane policer for a management-plane telemetry export changed one translator and the byte-size of the stored terms — nothing else.* | the bet of §3, the two toy translators, the seven-row table, and the real `translate()` emitting both call shapes in §4 |
| *Teardown removes exactly what enforcement installed and is idempotent; expiry and revocation may both trigger it, and the second trigger is a success, not an error.* | §2's double-teardown crash and its fix, replayed on the real mock in §4 |
| *A provider adding a third service writes one translator; the rest of the stack is already paid for.* | ✏️ 4 — you wrote the pinhole translator in a dozen lines and named the two rows it touches |
| *Delivery is caused by the entitlement's configuration: the router dials the buyer's collector because that config exists, and stops within one sample interval of teardown.* | §5's dial-out decision and its measured numbers (0 → flowing → 0) |

**Reviewer objections you can now answer:**
*"Isn't this just two demos glued together?"* — the claim is not that two services work;
it's the measured *difference* between them: five identical stages, two changed rows,
and (chapter 09) the only cost trace being terms byte-size in settlement gas (268k vs
448k). *"Is the policer enforcement real?"* — the shim is documented in the paper before
the reviewer finds it: committed config is the source of truth, the stand-in sits at the
router where the ASIC would, and it's absent on hardware. *"Is the telemetry real, or
just config?"* — the router dials out to the buyer's collector; a dial-in collector was
rejected precisely because it would prove nothing. *"Does invariance generalize beyond
these two?"* — argue it plausibly (any standing right on infrastructure: compute
reservations, storage leases, spectrum), claim it only for what was tested: two
services, one plane apart.

**Honesty inventory for §VI-B:** dataplane enforcement via the documented shim;
telemetry delivery real but its `sensor_paths` unenforced on the tunneled session
(gNSI pathz, not implemented); the lab's CPM return-path entry, absent on hardware;
invariance demonstrated for exactly two services.

*Next: [07a — Deploy an agent from zero](07a_deploy_an_agent_from_zero.ipynb)*